In [1]:
import torch

In [11]:
# Dense

x = torch.tensor([[1.,1.]]).T
w = torch.ones(2, 2, requires_grad=True)

l1 = torch.sum(w[:1] @ x)
l2 = torch.sum(w @ x)

l = 1/2 * (l1 + l2)

l.backward()

w.grad

tensor([[1.],
        [1.]])
tensor([[0.5000],
        [0.5000]])


In [12]:
# Dense

x = torch.tensor([[1.,1.]]).T
w_1 = torch.ones(2, 1, requires_grad=True)
w_2 = torch.ones(2, 1, requires_grad=True)

print(w_1.grad)
print(w_2.grad)

w = torch.cat([w_1, w_2], dim=1)
# w.retain_grad()

l1 = torch.sum(w_1.T @ x)
l2 = torch.sum(w @ x)

l = 1/2 * (l1 + l2)

l.backward()

# print(w.grad)
print(w_1.grad)
print(w_2.grad)

None
None
tensor([[1.],
        [1.]])
tensor([[0.5000],
        [0.5000]])


In [13]:
# Dense

x = torch.tensor([[1.,1.]])
w = torch.ones(2, 2, requires_grad=True)

# Let's say we want to use only the first column of the weight matrix. We can do that by slicing the weight matrix:
# Simulate a sgd with momentum for 2 epochs by hand:

v = torch.zeros_like(w)

for _ in range(3):

    l = torch.mean(w[:1].T @ x)
    
    print(w.grad)
    if w.grad is not None:
        w.grad.zero_()
    
    l.backward()
    
    v = 0.9 * v + 0.1 * w.grad
    
    with torch.no_grad():
        w.sub_(0.1 * v)

print(f"Updated weights: \n {w}")
print(f"Velocity: \n {v}")


Updated weights: 
 w1 
 tensor([[0.9579],
        [0.9579]], requires_grad=True), 
 w2 
 tensor([[0.9860],
        [0.9860]], requires_grad=True)
Velocity: 
 v1 
 tensor([[0.2033],
        [0.2033]])
Velocity: 
 v2 
 tensor([[0.0677],
        [0.0677]])


In [ ]:
# Now let's try to define the granularities with concatenations and do the same thing:

x = torch.tensor([[1.,1.]])
w_1 = torch.ones(2, 1, requires_grad=True)
w_2 = torch.ones(2, 1, requires_grad=True)

w = torch.cat([w_1, w_2], dim=1)

# Let's do the sgd steps with momentum again:

v_1 = torch.zeros_like(w_1)

for _ in range(3):

    l = torch.mean(w_1 @ x)

    if w_1.grad is not None:
        w_1.grad.zero_()

    l.backward()

    v_1 = 0.9 * v_1 + 0.1 * w_1.grad

    with torch.no_grad():
        w_1.sub_(0.1 * v_1)

print(f"Updated weights: \n w1 \n {w_1}, \n w2 \n {w_2}")
print(f"Velocity: \n v1 \n {v_1}")

Updated weights: 
 w1 
 tensor([[0.9719],
        [0.9719]], requires_grad=True), 
 w2 
 tensor([[1.],
        [1.]], requires_grad=True)
Velocity: 
 v1 
 tensor([[0.1355],
        [0.1355]])


In [44]:
print(f"Full linear weight: \n {lin.weight}")
print(f"Column 0 of lin.weight: \n {lin.weight[:, 0]}")
print(f"Column 1 of lin.weight: \n {lin.weight[:, 1]}")



Full linear weight: 
 Parameter containing:
tensor([[-0.0053,  0.3793],
        [-0.5820, -0.5204]], requires_grad=True)
Column 0 of lin.weight: 
 tensor([-0.0053, -0.5820], grad_fn=<SelectBackward0>)
Column 1 of lin.weight: 
 tensor([ 0.3793, -0.5204], grad_fn=<SelectBackward0>)


In [117]:
torch.manual_seed(0)
lin = torch.nn.Linear(2, 4, bias=False)

lin.weight[:2]

# F.linear(lin.weight[:,:1], x, None)

tensor([[-0.0053,  0.3793],
        [-0.5820, -0.5204]], grad_fn=<SliceBackward0>)

In [109]:
out_2

tensor([[ 0.3740],
        [-1.1024],
        [-0.0827],
        [ 0.5466]], grad_fn=<MmBackward0>)

In [8]:
out_small, out_full

(tensor([[-2.0796, -2.0041]], grad_fn=<AddmmBackward0>),
 tensor([[-2.1142, -2.0193]], grad_fn=<AddmmBackward0>))

In [14]:
import torch
import torch.nn.functional as F
torch.manual_seed(0)

# dims
H = 2   # hidden dim
I = 4   # full intermediate dim
k = 2   # prefix width (k < I)

x = torch.tensor([[1., 1.]])             # (1, H)
gate = torch.nn.Linear(H, I, bias=True)  # weight (I, H)
up   = torch.nn.Linear(H, I, bias=True)  # weight (I, H)
down = torch.nn.Linear(I, H, bias=True)  # weight (H, I)

opt = torch.optim.SGD(list(gate.parameters()) + list(up.parameters()) + list(down.parameters()), lr=0.1, momentum=0.9)

for _ in range(10):
    opt.zero_grad()

    # small branch: slice rows of gate/up -> (k, H) -> produces (1, k)
    gate_w = gate.weight[:k]                 # (k, H)
    gate_b = gate.bias[:k] if gate.bias is not None else None
    up_w   = up.weight[:k]                   # (k, H)
    up_b   = up.bias[:k] if up.bias is not None else None

    a_small = F.silu(F.linear(x, gate_w, gate_b))   # (1, k)
    b_small = F.linear(x, up_w, up_b)               # (1, k)
    intermediate_small = a_small * b_small         # (1, k)

    # project back to full hidden dim using down.weight[:, :k] -> (H, k)
    out_small = F.linear(intermediate_small, down.weight[:, :k], down.bias)  # (1, H)

    # full branch: compute full intermediate (1, I) and project -> (1, H)
    a_full = F.silu(F.linear(x, gate.weight, gate.bias))  # (1, I)
    b_full = F.linear(x, up.weight, up.bias)              # (1, I)
    intermediate_full = a_full * b_full                   # (1, I)
    out_full = F.linear(intermediate_full, down.weight, down.bias)  # (1, H)

    out = 0.5 * (out_small + out_full)   # both are (1, H), no padding needed
    loss = out.mean()
    loss.backward()
    opt.step()

print("out shape:", out.shape)  # (1, H)
print("out:", out)


out shape: torch.Size([1, 2])
out: tensor([[-4.8926, -8.0827]], grad_fn=<MulBackward0>)


In [19]:
[g for g in gate_blocks]

[Linear(in_features=2, out_features=2, bias=True),
 Linear(in_features=2, out_features=2, bias=True)]

In [16]:
# Blocked + concatenated version of the same loop

import torch
import torch.nn.functional as F

torch.manual_seed(0)

# dims
H = 2   # hidden dim
I = 4   # full intermediate dim
k = 2   # block width

assert I % k == 0, "I must be divisible by k for block concatenation"
num_blocks = I // k

x = torch.tensor([[1., 1.]])  # (1, H)

# Independent blocks: each block has its own parameters, and the full weight is built by concatenation.
gate_blocks = torch.nn.ModuleList([torch.nn.Linear(H, k, bias=True) for _ in range(num_blocks)])
up_blocks = torch.nn.ModuleList([torch.nn.Linear(H, k, bias=True) for _ in range(num_blocks)])
down_blocks = torch.nn.ModuleList([torch.nn.Linear(k, H, bias=False) for _ in range(num_blocks)])
down_bias = torch.nn.Parameter(torch.zeros(H))

params = (
    list(gate_blocks.parameters())
    + list(up_blocks.parameters())
    + list(down_blocks.parameters())
    + [down_bias]
)
opt = torch.optim.SGD(params, lr=0.1, momentum=0.9)

for _ in range(10):
    opt.zero_grad()

    # small branch: use the first block only
    a_small = F.silu(gate_blocks[0](x))      # (1, k)
    b_small = up_blocks[0](x)                # (1, k)
    intermediate_small = a_small * b_small   # (1, k)
    out_small = F.linear(intermediate_small, down_blocks[0].weight, down_bias)  # (1, H)

    # full branch: concatenate all independent blocks
    gate_full = torch.cat([block(x) for block in gate_blocks], dim=-1)  # (1, I)
    up_full = torch.cat([block(x) for block in up_blocks], dim=-1)      # (1, I)
    intermediate_full = F.silu(gate_full) * up_full                     # (1, I)
    down_full_weight = torch.cat([block.weight for block in down_blocks], dim=1)  # (H, I)
    out_full = F.linear(intermediate_full, down_full_weight, down_bias)            # (1, H)

    out = 0.5 * (out_small + out_full)
    loss = out.mean()
    loss.backward()
    opt.step()

print("out shape:", out.shape)
print("out:", out)


out shape: torch.Size([1, 2])
out: tensor([[-3.6736, -7.2727]], grad_fn=<MulBackward0>)


In [17]:
# Concatenated (granular) experiment using two small linear layers + SGD(momentum)

torch.manual_seed(0)

x = torch.tensor([[1., 1.]])
lin1 = torch.nn.Linear(1, 2, bias=False)
lin2 = torch.nn.Linear(1, 2, bias=False)

opt = torch.optim.SGD(list(lin1.parameters()) + list(lin2.parameters()), lr=0.1, momentum=0.9)

for _ in range(10000):

    opt.zero_grad()
    
    # Only use the first branch in the output for small granularity
    out_1 = (lin1.weight * x).T

    # Use the whole model for large granularity
    combined_weight = torch.cat([lin1.weight, lin2.weight], dim=1)
    out_2 = F.linear(x, combined_weight, None)

    out = 1/2 * (out_1 + out_2)

    loss = out.mean()
    loss.backward()
    opt.step()

print('combined_weight:')
print(torch.cat([lin1.weight, lin2.weight], dim=1))
print('lin1.weight:')
print(lin1.weight)
print('lin2.weight:')
print(lin2.weight)
print('momentum buffer (lin1):')
print(opt.state.get(lin1.weight, {}).get('momentum_buffer'))
print('momentum buffer (lin2):')
print(opt.state.get(lin2.weight, {}).get('momentum_buffer'))

combined_weight:
tensor([[-4995.5073, -2498.5730],
        [-4994.9634, -2498.4858]], grad_fn=<CatBackward0>)
lin1.weight:
Parameter containing:
tensor([[-4995.5073],
        [-4994.9634]], requires_grad=True)
lin2.weight:
Parameter containing:
tensor([[-2498.5730],
        [-2498.4858]], requires_grad=True)
momentum buffer (lin1):
tensor([[5.0000],
        [5.0000]])
momentum buffer (lin2):
tensor([[2.5000],
        [2.5000]])
